# 컨텍스트 로트

근거 문장의 위치와 주변 문장 수가 답에 어떤 차이를 만들까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 로컬 생성 모델 준비

고정 revision의 사전학습 모델과 tokenizer를 로컬에서 불러옵니다. `generate()`의 입력은 프롬프트 문자열과 생성 설정입니다. 함수 안에서 문자열을 토큰 ID로 바꾸고 모델이 새 토큰 ID를 생성한 뒤, 새 부분만 문자열로 복원해 반환합니다. 모델 구조를 구현하거나 학습하는 셀이 아닙니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, logging, set_seed

logging.set_verbosity_error()

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION)
model.eval()
print("model:", MODEL_ID)
print("parameters:", f"{model.num_parameters():,}")

model: Qwen/Qwen2.5-0.5B-Instruct
parameters: 494,032,768


In [2]:
def generate(prompt: str, *, temperature: float = 0.7, seed: int = 0, max_new_tokens: int = 48) -> str:
    # 채팅 템플릿을 적용한 뒤 새로 생성된 토큰만 복원합니다.
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    set_seed(seed)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature if temperature > 0 else None,
            top_p=0.9 if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 위치와 길이를 바꾸는 실험

정답 근거, 질문, 모델, 생성 방식은 고정합니다. 무관한 문장 수를 2·20·60개로 바꾸고 근거 위치를 시작·중간·끝으로 바꿉니다. 각 조건의 반환 문자열에 `314159`가 있으면 적중으로 셉니다. `hits=0/2`는 두 중복 실행 모두 코드 문자열을 반환하지 않았다는 뜻이며, 모델 전체의 실패율을 뜻하지 않습니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
fact = "해바라기 작전의 확인 코드는 314159입니다."
filler = "주문 조회 화면에는 결제 상태와 배송 상태가 표시됩니다."
question = "해바라기 작전의 확인 코드만 답하세요."
conditions = []
for repeats in [2, 20, 60]:
    for position in ["start", "middle", "end"]:
        items = [filler] * repeats
        index = {"start": 0, "middle": repeats // 2, "end": repeats}[position]
        items.insert(index, fact)
        conditions.append((repeats, position, "\n".join(items) + "\n\n" + question))

observations = []
for repeats, position, prompt in conditions:
    outputs = [generate(prompt, temperature=0, seed=seed, max_new_tokens=20) for seed in [1, 2]]
    hits = sum("314159" in output for output in outputs)
    messages = [{"role": "user", "content": prompt}]
    templated = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = len(tokenizer.encode(templated))
    observations.append({"repeats": repeats, "position": position, "tokens": tokens, "outputs": outputs})
    print(f"filler={repeats:2} position={position:6} tokens={tokens:4} hits={hits}/2 outputs={outputs}")

filler= 2 position=start  tokens=  99 hits=2/2 outputs=['314159', '314159']


filler= 2 position=middle tokens=  99 hits=0/2 outputs=['결제 상태: 완료\n배송 상태: 완료', '결제 상태: 완료\n배송 상태: 완료']


filler= 2 position=end    tokens=  99 hits=0/2 outputs=['결제 상태: 완료\n배송 상태: 완료', '결제 상태: 완료\n배송 상태: 완료']


filler=20 position=start  tokens= 441 hits=2/2 outputs=['해바라기 작전의 확인 코드는 314159입니다.', '해바라기 작전의 확인 코드는 314159입니다.']


filler=20 position=middle tokens= 441 hits=0/2 outputs=['주문 조회 화면에는 결제 상태와 배송 상태가 표시되지 않습니다.', '주문 조회 화면에는 결제 상태와 배송 상태가 표시되지 않습니다.']


filler=20 position=end    tokens= 441 hits=2/2 outputs=['해바라기 작전의 확인 코드는 314159입니다.', '해바라기 작전의 확인 코드는 314159입니다.']


filler=60 position=start  tokens=1201 hits=2/2 outputs=['314159', '314159']


filler=60 position=middle tokens=1201 hits=0/2 outputs=['주문 조회 화면에는 결제 상태와 배송 상태가 표시됩니다.', '주문 조회 화면에는 결제 상태와 배송 상태가 표시됩니다.']


filler=60 position=end    tokens=1201 hits=2/2 outputs=['해바라기 작전의 확인 코드는 314159입니다.', '해바라기 작전의 확인 코드는 314159입니다.']


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [4]:
assert len(observations) == 9
assert all(len(item["outputs"]) == 2 for item in observations)
print("검증: 3개 길이 × 3개 위치의 중복 출력을 확인했습니다.")

검증: 3개 길이 × 3개 위치의 중복 출력을 확인했습니다.
